NB1: Baseline

Our question: Does the improvement in generalisation performance of neural networks when using dropout regularisation hold with increasing training dataset sizes? ^^ think we use generalisation gap instead of performance

Theory Claim: For smaller trained modules, dropout reduces overfitting, increasing overall generalisation performance/the generalisation gap - should verify which is true to the claim.

Why choose 10%? Is this reasonable base to show overfitting? Best results were at 2.5%...

Our Baseline: Overffited dataset with increasingly applied levels of dropout. 

Run with over 5 random seeds. #These are currently fixed seeds

Fashion-MNIST (70,000 grayscale images across 10 clothing categories).
- Fixed image sizes of 28x28 pixels.
- Up to 60,000 training images allows for a range in test cases when testing the breaking point of the claim
- Clothing categories are visually similar making classification more difficult

Need to code: function to train fashion-MNIST and then apply with dropout at difference percentages. 


Might need to pip install torch and pip install torchvision

In [79]:
%pip install torch
%pip install torchvision

#libray imports
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import datasets, transforms

#seed
#torch.manual_seed(16)

device = ('cuda' if torch.cuda.is_available() else
          'mps' if torch.backends.mps.is_available() else 'cpu')

transform = transforms.Compose([transforms.ToTensor()])
train_dataset = datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
print(f'FashionMNIST: {len(train_dataset)} train images, {len(test_dataset)} test images')

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: C:\Users\holly\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
FashionMNIST: 60000 train images, 10000 test images



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: C:\Users\holly\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


256 hidden units gives:
- Input Layer: 784 neurons
- Hidden Layer 1: 256 neurons
- Hidden Layer 2: 256 neurons
- Output Layer: 10 neurons

This layout is complex enough to learn patterns in Fashion-MNIST.

In [80]:
#Settings
HIDDEN_UNITS = 256
TRAIN_PERCENT = 0.05
N_TRAIN = int(len(train_dataset)* TRAIN_PERCENT)
EPOCHS = 20 #We need to work out if we want all epochs no batches or all batch no epoch or a mix (mini-batch) - may need to increase epochs to allow model to show overfitting better (20 - 30)
BATCH = 128 #keep mini-batch (balance between computation efficient and gradient stability)
LEARNING_RATE = 0.001 #keep
DROPOUT_RATES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
SEED = [16, 31, 9, 92, 2] #keep 5

In [ ]:
#Loading the dataset into device tensors: 10% of the dataset will be used for training

def to_tensor(dataset, index=None):
    images, labels = (dataset.data, dataset.targets) if index is None else (dataset.data[index], dataset.targets[index])
    X = images.float().div(255).view(-1,784)
    return X.to(device), labels.to(device)

#generator = torch.Generator().manual_seed(16)
def random_sample(train_dataset, test_dataset):
    order = torch.randperm(len(train_dataset))[:N_TRAIN]
    x_train, y_train = to_tensor(train_dataset, order)
    x_test, y_test = to_tensor(test_dataset)
    #print(f"device {device} | train {tuple(x_train.shape)} | test {tuple(x_test.shape)}")
    print(f"Training images: {x_train.shape}")
    print(f"Training labels: {y_train.shape}")
    print(f"Testing images: {x_test.shape}")
    print(f"Testing labels: {y_test.shape}")
    return x_train, x_test, y_train, y_test

In [82]:
#Making and training the model

#Why did we choose 2 hidden layers? we could change this ..., standardise this ... or explain why 2 is the best option.
def make_model(dropout_rate):
    return torch.nn.Sequential(
        torch.nn.Linear(784, HIDDEN_UNITS), torch.nn.ReLU(), torch.nn.Dropout(dropout_rate),
        torch.nn.Linear(HIDDEN_UNITS, HIDDEN_UNITS), torch.nn.ReLU(), torch.nn.Dropout(dropout_rate),
        torch.nn.Linear(HIDDEN_UNITS, 10) #There are only 10 types of clothing class in fashionMNIST.
    ).to(device)

@torch.no_grad()
def accuracy(model, X, y):
    model.eval()
    return(model(X).argmax(1) == y).float().mean().item()

@torch.no_grad()
def calculate_loss(model, X, y):
    model.eval()
    loss_function = nn.CrossEntropyLoss()
    return loss_function(model(X), y).item()

def train(model, x_train, y_train, x_test, y_test, seed, verbose=True):
    random_generator = torch.Generator(device="cpu").manual_seed(seed)
    adam_opt = torch.optim.Adam(model.parameters(), lr = LEARNING_RATE)
    loss_function = nn.CrossEntropyLoss()
    results = {"train": [], 
               "test": [],
               "train_loss": [],
               "test_loss": []}
    for ep in range(EPOCHS):
        model.train()
        shuffle = torch.randperm(len(x_train), generator=random_generator).to(device)
        for i in range(0, len(x_train), BATCH):
            index = shuffle[i:i + BATCH]
            adam_opt.zero_grad()
            loss = loss_function(model(x_train[index]), y_train[index])
            loss.backward()
            adam_opt.step()
        results["train"].append(accuracy(model, x_train, y_train))
        results["test"].append(accuracy(model, x_test, y_test))
        results["train_loss"].append(calculate_loss(model, x_train, y_train))
        results["test_loss"].append(calculate_loss(model, x_test, y_test))
        if verbose and (ep + 1) % 10  ==0:
            print(
                f"epoch {ep + 1} | "
                f"Training Accuracy: {results['train'][-1]:.4f} | "
                f"Testing Accuracy: {results['test'][-1]:.4f} | "
                f"Training loss: {results['train_loss'][-1]:.4f} | "
                f"Testing Loss: {results['test_loss'][-1]:.4f}"
                
            )
    return results

In [83]:
#Run with varying dropout
def dropout_experiment(seed):
    models = {}
    results = {}
    x_train, x_test, y_train, y_test = random_sample(train_dataset, test_dataset)

    for r in DROPOUT_RATES:
        models[r] = make_model(r)
        results[r] = train(models[r], x_train, y_train, x_test, y_test, seed)

    print(f"{'Dropout Rate'} {'Training Accuracy'} {'Test Accuracy'} {'Difference'} {'Training Loss'} {'Testing Loss'}")
    for r in DROPOUT_RATES:
        train_data = results[r]["train"][-1]
        test_data = results[r]["test"][-1]
        train_loss = results[r]["train_loss"][-1]
        test_loss = results[r]["test_loss"][-1]
        print(f"{r:12.2f} {train_data:17.4f} {test_data:13.4f} {train_data-test_data:10.4f} {train_loss:12.4f} {test_loss:13.4f}")

In [84]:
def main():
    for i in SEED:
        print(f"\nSeed: {i}")
        dropout_experiment(i)

main()


Seed: 16
Training images: torch.Size([3000, 784])
Training labels: torch.Size([3000])
Testing images: torch.Size([10000, 784])
Testing labels: torch.Size([10000])
epoch 10 | Training Accuracy: 0.8740 | Testing Accuracy: 0.8177 | Training loss: 0.3432 | Testing Loss: 0.5278
epoch 20 | Training Accuracy: 0.9277 | Testing Accuracy: 0.8265 | Training loss: 0.2039 | Testing Loss: 0.5400
epoch 10 | Training Accuracy: 0.8633 | Testing Accuracy: 0.8072 | Training loss: 0.3611 | Testing Loss: 0.5388
epoch 20 | Training Accuracy: 0.9320 | Testing Accuracy: 0.8335 | Training loss: 0.1989 | Testing Loss: 0.5075
epoch 10 | Training Accuracy: 0.8737 | Testing Accuracy: 0.8205 | Training loss: 0.3579 | Testing Loss: 0.5122
epoch 20 | Training Accuracy: 0.9277 | Testing Accuracy: 0.8391 | Training loss: 0.2102 | Testing Loss: 0.4906
epoch 10 | Training Accuracy: 0.8607 | Testing Accuracy: 0.8095 | Training loss: 0.3748 | Testing Loss: 0.5168
epoch 20 | Training Accuracy: 0.9070 | Testing Accuracy: 0.

VARIABILITY ANALYSIS
